In [1]:
import torch
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import (
    CategoricalConditioningConfig,
    TimeConditioningConfig,
)
from src.monitoring.utils import fieldplot, pca_project, save_go_detached, scatterplot
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig
from src.priors.unimodal import UnimodalGaussianPriorConfig
from src.stochastic_chart_transport.constraint import (
    ChartPretrainConfig,
    IntegratedChartConstraintConfig,
    LatentNormAnchorConfig,
    LatentScaleAnchorConfig,
    ReconstructionConfig,
)
from src.stochastic_chart_transport.critic import CriticLossConfig
from src.stochastic_chart_transport.fibers import FlatFiberPacking
from src.stochastic_chart_transport.model import ChartTransportModelConfig
from src.stochastic_chart_transport.study import (
    StochasticChartTransportStudyConfig,
    StochasticChartTransportStudyState,
)
from src.stochastic_chart_transport.transport import StochasticChartTransportLossConfig

device = "cuda:1"
num_modes = 8
batch_size = 2048
ambient_dimension = 3
fiber_dimension = ambient_dimension
hidden_dim = 512
hidden_dims = [hidden_dim] * 4

/tmp/ipykernel_534470/3498108122.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
mc = ChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension + fiber_dimension]
        + hidden_dims
        + [ambient_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension]
        + hidden_dims
        + [ambient_dimension + fiber_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [ambient_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
        cat_conditioning_config=CategoricalConditioningConfig(
            num_classes=2,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)

data_scale = 10.0
latent_scale = 1.0

data_reconstruction_config = ReconstructionConfig(
    huber_delta=10.0,
    weight=5.0
)
integrated_anchor_config = LatentScaleAnchorConfig(
    latent_norm_weight=0.0,
    latent_zero_mean_weight=1.0,
    target_norm_per_dimension=latent_scale,
)
model_reconstruction_config = ReconstructionConfig(
    huber_delta=10.0,
    weight=1.0
)

pretrain_config = ChartPretrainConfig(
    data_reconstruction=data_reconstruction_config,
    model_reconstruction=ReconstructionConfig(
        huber_delta=10.0,
        weight=0.0
    ),
    data_fiber_reconstruction=ReconstructionConfig(
        huber_delta=10.0,
        weight=0.1
    ),
    prior_reconstruction=ReconstructionConfig(
        huber_delta=10.0,
        weight=1.0
    ),
    anchor_config=LatentNormAnchorConfig(latent_norm_weight=1e-3)
)

transport_config = StochasticChartTransportLossConfig(
    transport_step_multiplier=0.05,
    transport_step_cap=0.05,
    num_time_samples=5,
    t_range=(0.01, 0.99),
    antipodal_estimate=True,
    # Weight specifications
    decoder_huber_delta=5.0,
    forward_kl_weight=10.0,
    reverse_kl_weight=10.0,
    data_decoder_weight_multiplier=1.0
)

integrated_constraint_config = IntegratedChartConstraintConfig(
    data_reconstruction=data_reconstruction_config,
    model_reconstruction=ReconstructionConfig(
        huber_delta=10.0,
        weight=0.0
    ),
    data_fiber_reconstruction=ReconstructionConfig(
        huber_delta=10.0,
        weight=0.1
    ),
    data_latent_anchor=integrated_anchor_config,
    model_latent_anchor=integrated_anchor_config
)

study_config = StochasticChartTransportStudyConfig(
    data=MultimodalGaussianDataConfig.initialize(
        num_modes=8, mode_std=0.05, ambient_dimension=ambient_dimension, offset=0,
        scale=data_scale,
    ),
    prior=AnchoredGaussianScaleMixturePriorConfig(
        latent_shape=[ambient_dimension], precision=1.5, scale=latent_scale,
    ),
    model=mc,
    pretrain=pretrain_config,
    critic=CriticLossConfig(huber_delta=10.0, weight=1.0, t_min=0.005, t_max=0.99),
    transport=transport_config,
    integrated_constraint=integrated_constraint_config,
    fiber_packing=FlatFiberPacking(fiber_ndims=fiber_dimension),
)
display(study_config.visualize())
state = StochasticChartTransportStudyState.initialize(
    config=study_config, device=torch.device(device)
)

### Monitoring

In [3]:
import plotly.graph_objects as go
from jaxtyping import Float
from plotly.subplots import make_subplots
from torch import Tensor

visualize_batch_size_per_mode = 512
canonical_samples = {
    str(j): study_config.data.sample_class(
        mode_id=j, batch_size=visualize_batch_size_per_mode
    ).to(device)
    for j in range(num_modes)
}
canonical_prior = state.prior_config.sample(batch_size=visualize_batch_size_per_mode).to(
    device
)
canonical_fiber = torch.randn_like(canonical_prior)

def monitor(state):
    with torch.no_grad():
        model_sample = state.decode(canonical_prior)[0]
        model_latent = state.encode(data=model_sample, fiber=canonical_fiber)
        scatter_data_latent = {
            k: state.encode(
                data=v,
                fiber=canonical_fiber,
            )
            for k, v in canonical_samples.items()
        }
        scatter_model_latent = {"model": model_latent}
    data_scatterplot_dict = dict(scatter_data_latent)
    model_scatterplot_dict = dict(scatter_model_latent)

    # Vector fields
    with torch.no_grad():
        data_transport_field = {
            k: state.config.transport._estimate_transport_fields(
                state,
                data_latent=v,
                model_latent=v
            )[0]
            for k, v in scatter_data_latent.items()
        }
        model_transport_field = {
            "model": state.config.transport._estimate_transport_fields(
                state,
                data_latent=model_latent,
                model_latent=model_latent
            )[1]
        }
    conditioned_data_transport_field = {
        k: state.config.transport._scale_clip_transport_field(
            v
        )
        for k, v in data_transport_field.items()
    }
    conditioned_model_transport_field = {
        "model": state.config.transport._scale_clip_transport_field(
            model_transport_field["model"]
        )
    }

    data_scatter = scatterplot(data_scatterplot_dict | model_scatterplot_dict)
    model_scatter = scatterplot(
        canonical_samples | {"model": model_sample}
    )
    data_field = fieldplot(base=data_scatterplot_dict, field=data_transport_field)
    model_field = fieldplot(base=model_scatterplot_dict, field=model_transport_field)
    data_conditioned_field = fieldplot(base=data_scatterplot_dict, field=conditioned_data_transport_field)
    model_conditioned_field = fieldplot(base=model_scatterplot_dict, field=conditioned_model_transport_field)

    # display(field)
    # display(conditioned_field)

    combo = make_subplots(
        rows=2,
        cols=3,
        specs=([[{"type": "scene"}]*3]*2)
    )
    for j, fig in enumerate([data_scatter, data_field, data_conditioned_field, model_scatter, model_field, model_conditioned_field]):
        for tr in fig.data:
            combo.add_trace(
                tr, row=j // 3 + 1, col=j % 3 +1
            )
    return combo

### Chart and critic pretrain

In [ ]:
if not Path("artifacts/state.pkl").exists():
    pbar = tqdm(range(1000))

    for p in pbar:
        data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

        loss = state.compute_chart_pretrain_loss(data=data)

        total_loss = loss.sum()
        total_loss.backward()
        state.step_and_zero_grad()
        pbar.set_postfix({"loss": total_loss.item()})

    pbar = tqdm(range(1_000))
    for p in pbar:
        data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

        loss = state.compute_critic_only_loss(data=data)
        loss = loss.sum()
        loss.backward()
        state.step_and_zero_grad()

        pbar.set_postfix({"loss": loss.item()})

    torch.save(state, "artifacts/state.pkl")
    monitor(state)

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
state = torch.load("artifacts/state.pkl", weights_only=False)

## Integrated transport

In [ ]:
pbar = tqdm(range(100_000))

transport_chart_every_n_steps = 5
for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    losses = state.compute_integrated_loss(
        data=data,
        compute_transport_loss=(p % transport_chart_every_n_steps == 0),
    )

    total_loss = losses.sum()
    total_loss.backward()
    state.step_and_zero_grad()

    pbar.set_postfix({"loss": total_loss.item()})
    if p % transport_chart_every_n_steps == 0:
        if p % 200 == 0:
            fig = monitor(state)
            save_go_detached(fig, folder="artifacts", name=str(p // 5))

  0%|          | 0/100000 [00:00<?, ?it/s]